# 数据库实习三：基于 SQL 的数据分析实习

本报告核心计算全部由 SQLite SQL 脚本完成，Python 使用内置 `sqlite3` 运行本地数据库，Jupyter Notebook 只读取 `results/` 目录中导出的 CSV 文件，负责结果展示、可视化和分析结论。

完成的任务：

- 任务一：基于 SQL 的数据预处理。
- 任务二：MovieLens 数据集分析查询。
- 任务三：世界幸福指数数据集熵分析。

## 0. 环境准备与结果文件检查

运行本 Notebook 前，应先执行：

```bash
python scripts/run_sqlite_pipeline.py
```

SQLite 数据库文件会生成在 `data/dbproj3.sqlite`。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid", font="Microsoft YaHei")

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "reports" else cwd
REPORT_DIR = PROJECT_ROOT / "reports"
RESULTS_DIR = PROJECT_ROOT / "results"

csv_files = {
    "task1_before_after": "task1_before_after.csv",
    "task1_quality_summary": "task1_quality_summary.csv",
    "task2_dataset_summary": "task2_dataset_summary.csv",
    "task2_top_movies": "task2_top_movies.csv",
    "task2_genre_top_movies": "task2_genre_top_movies.csv",
    "task2_user_top_rating_genres": "task2_user_top_rating_genres.csv",
    "task2_user_top_watch_genres": "task2_user_top_watch_genres.csv",
    "task2_similar_users": "task2_similar_users.csv",
    "task3_entropy_metrics": "task3_entropy_metrics.csv",
    "task3_feature_importance": "task3_feature_importance.csv",
    "task3_joint_metrics": "task3_joint_metrics.csv",
}

missing = [name for name in csv_files.values() if not (RESULTS_DIR / name).exists()]
if missing:
    print("以下结果文件尚未生成，请先执行 python scripts/run_sqlite_pipeline.py：")
    for name in missing:
        print(" -", RESULTS_DIR / name)
else:
    print("所有结果文件均已找到。")

In [ ]:
def read_result(key: str) -> pd.DataFrame:
    path = RESULTS_DIR / csv_files[key]
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)


dfs = {key: read_result(key) for key in csv_files}
{key: df.shape for key, df in dfs.items()}

## 1. 数据库表结构与脚本分工

- `sql/00_create_database.sql`：创建数据库、任务一用户表、MovieLens 表、幸福指数 raw 表和标准表。
- `sql/task1_preprocessing.sql`：插入脏数据，完成清洗，并生成 `v_task1_before_after` 与 `v_task1_quality_summary`。
- `sql/task2_movielens_analysis.sql`：拆分电影类型，完成五类 MovieLens 查询，并生成任务二结果视图。
- `sql/task3_happiness_entropy.sql`：统一 2015-2019 幸福指数字段，分桶后计算熵、信息增益和互信息。
- `scripts/run_sqlite_pipeline.py`：使用 Python 内置 sqlite3 创建数据库、导入 CSV、执行 SQL 并导出结果 CSV。
- `scripts/export_results.py`：从已经生成的 SQLite 数据库重新导出 CSV。

## 2. 任务一：SQL 数据预处理

任务一在原始要求的姓名、手机号、邮箱、注册时间、地址、年龄、备注清洗基础上，新增了 5 项扩展规则：中文姓名合法性校验、手机号段校验、邮箱域名提取、注册日期有效区间校验、地址城市提取。

In [ ]:
task1 = dfs["task1_before_after"]
task1_quality = dfs["task1_quality_summary"]

display(task1.head(20))
display(task1_quality)

In [ ]:
if not task1_quality.empty:
    fig, ax = plt.subplots(figsize=(9, 4))
    sns.barplot(data=task1_quality, x="metric", y="value", ax=ax, color="#4c78a8")
    ax.set_title("任务一：清洗后异常/缺失指标统计")
    ax.set_xlabel("指标")
    ax.set_ylabel("数量")
    ax.tick_params(axis="x", rotation=35)
    plt.tight_layout()
    plt.show()

要点：

- SQL 中使用 `REGEXP_REPLACE()`、`REGEXP`、`SAFE_DATE()`、`CASE WHEN` 完成清洗、解析和校验；这些扩展函数由 Python 的 `sqlite3` 注册。
- 结果表同时保留原始字段与标准化字段，便于形成清洗前后对比。
- 扩展字段 `email_domain`、`city_name` 让清洗结果可以继续用于统计分析。

## 3. 任务二：MovieLens 分析查询

MovieLens 查询围绕电影评分、电影类型、用户偏好和相似用户展开。为了避免样本量过小造成偶然高分，Top 电影查询加入了最低评分次数阈值。

In [ ]:
for key in [
    "task2_dataset_summary",
    "task2_top_movies",
    "task2_genre_top_movies",
    "task2_user_top_rating_genres",
    "task2_user_top_watch_genres",
    "task2_similar_users",
]:
    print("\n", key)
    display(dfs[key].head(20))

In [ ]:
top_movies = dfs["task2_top_movies"]
if not top_movies.empty:
    plot_df = top_movies.sort_values("avg_rating", ascending=True)
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.barplot(data=plot_df, x="avg_rating", y="title", ax=ax, color="#59a14f")
    ax.set_title("任务二：平均评分前 10 的电影")
    ax.set_xlabel("平均评分")
    ax.set_ylabel("电影")
    plt.tight_layout()
    plt.show()

In [ ]:
genre_top = dfs["task2_genre_top_movies"]
if not genre_top.empty:
    genre_counts = genre_top.groupby("genre", as_index=False).size()
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(data=genre_counts, x="genre", y="size", ax=ax, color="#f28e2b")
    ax.set_title("任务二：各类型进入 Top10 结果的电影数量")
    ax.set_xlabel("类型")
    ax.set_ylabel("数量")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()

要点：

- `movie_genres` 将多类型字段拆成一行一个类型，使类型维度查询更自然。
- 每个类型 Top10 和每个用户 Top5 都使用窗口函数 `ROW_NUMBER()` 实现。
- 相似用户对同时考虑共同观影类型数量和评分差标准差，比单纯共同看过电影更稳健。

## 4. 任务三：幸福指数熵分析

幸福指数原始数据 2015-2019 年字段名不完全一致，SQL 先统一到 `happiness` 表，再用 `NTILE(3)` 对连续指标按年份分桶。分类标签 `score_level` 按分数划分为 high、middle、low。

In [ ]:
entropy_metrics = dfs["task3_entropy_metrics"]
feature_importance = dfs["task3_feature_importance"]
joint_metrics = dfs["task3_joint_metrics"]

display(entropy_metrics)
display(feature_importance.head(30))
display(joint_metrics.head(20))

In [ ]:
if not feature_importance.empty:
    all_scope = feature_importance[feature_importance["scope_year"].astype(str) == "all"].copy()
    all_scope = all_scope.sort_values("information_gain", ascending=True)
    fig, ax = plt.subplots(figsize=(9, 4))
    sns.barplot(data=all_scope, x="information_gain", y="feature_name", ax=ax, color="#e15759")
    ax.set_title("任务三：全样本特征信息增益排名")
    ax.set_xlabel("信息增益")
    ax.set_ylabel("特征")
    plt.tight_layout()
    plt.show()

In [ ]:
if not feature_importance.empty:
    yearly = feature_importance[feature_importance["scope_year"].astype(str) != "all"].copy()
    if not yearly.empty:
        pivot = yearly.pivot_table(
            index="feature_name",
            columns="scope_year",
            values="information_gain",
            aggfunc="mean",
        )
        fig, ax = plt.subplots(figsize=(9, 5))
        sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGnBu", ax=ax)
        ax.set_title("任务三：不同年份特征信息增益变化")
        ax.set_xlabel("年份")
        ax.set_ylabel("特征")
        plt.tight_layout()
        plt.show()

要点：

- 标签熵 `H(Y)` 衡量 high/middle/low 幸福类别的混乱程度。
- 条件熵 `H(Y|X)` 越低，说明给定某指标分桶后，幸福类别越容易被区分。
- 信息增益 `IG = H(Y) - H(Y|X)` 用于比较 GDP、社会支持、健康寿命、自由、慷慨、腐败感知等特征的重要性。
- 联合特征互信息用于观察两个因素组合后是否比单个因素更能解释幸福等级。

## 5. 总结

本项目将 SQL 用于完整的数据分析流程：任务一完成脏数据清洗和标准化；任务二在 MovieLens 数据集上完成多维分析查询；任务三将机器学习中的熵、条件熵、信息增益、基尼系数和互信息用 SQL 落地到真实幸福指数数据。

采用 SQL 负责计算、Notebook 负责展示的方式，可以同时保证数据库实习的 SQL 实现重点和最终报告的可读性。